# Execute a workflow with pypushflow

This page presents benchmarks of the execution of a Ewoks workflow with and without pypushplow (`ppf`) engine.

## Initial setup

Install *ewoks[ppf]* and *numpy*

In [ ]:
!pip install ewoks[ppf]
!pip install numpy

## Task and workflow definition

For the demonstration, let's create a workflow with two branchs doing each a simple matrix operation:
- Branch 1 will generate a matrix and transpose it
- Branch 2 will generate a matrix and flip it

Both branches will join in a single node that will multiply the resulting matrices.

Below, we define the Ewoks tasks needed for the workflow

In [ ]:
from ewokscore import Task
import numpy


class MatrixGeneration(
    Task, input_names=["rows", "cols", "fill"], output_names=["matrix"]
):
    def run(self):
        rows = self.inputs.rows
        cols = self.inputs.cols
        val = self.inputs.fill
        self.outputs.matrix = numpy.full((rows, cols), val)


class MatrixTranspose(Task, input_names=["M"], output_names=["Mt"]):
    def run(self):
        M = self.inputs.M
        self.outputs.Mt = M.transpose()


class MatrixVerticalFlip(Task, input_names=["M"], output_names=["Mf"]):
    def run(self):
        M = self.inputs.M

        self.outputs.Mf = numpy.flip(M, 0)


class MatrixMultiplication(Task, input_names=["A", "B"], output_names=["C"]):
    """C = A * B"""

    def run(self):
        A = self.inputs.A
        B = self.inputs.B
        self.outputs.C = A @ B

Now, we create a workflow out of this task:

```
Generation matrix A -> Transpose 
                                \
                                 Matrix multiplication
                                /
Generation matrix B ------> Flip
```

In [ ]:
nodes = [
    {
        "id": "matrixGenerationA",
        "task_identifier": "__main__.MatrixGeneration",
        "task_type": "class",
    },
    {
        "id": "matrixTransposeA",
        "task_identifier": "__main__.MatrixTranspose",
        "task_type": "class",
    },
    {
        "id": "matrixGenerationB",
        "task_identifier": "__main__.MatrixGeneration",
        "task_type": "class",
    },
    {
        "id": "matrixVerticalFlipB",
        "task_identifier": "__main__.MatrixVerticalFlip",
        "task_type": "class",
    },
    {
        "id": "matrixMultiplication",
        "task_identifier": "__main__.MatrixMultiplication",
        "task_type": "class",
    },
]

links = [
    {
        "source": "matrixGenerationA",
        "target": "matrixTransposeA",
        "data_mapping": [{"source_output": "matrix", "target_input": "M"}],
    },
    {
        "source": "matrixTransposeA",
        "target": "matrixMultiplication",
        "data_mapping": [{"source_output": "Mt", "target_input": "A"}],
    },
    {
        "source": "matrixGenerationB",
        "target": "matrixVerticalFlipB",
        "data_mapping": [{"source_output": "matrix", "target_input": "M"}],
    },
    {
        "source": "matrixVerticalFlipB",
        "target": "matrixMultiplication",
        "data_mapping": [{"source_output": "Mf", "target_input": "B"}],
    },
]

workflow = {
    "graph": {"id": "parallelMatrixWorkflow"},
    "nodes": nodes,
    "links": links,
}


## Parallel execution with pypushflow against standard Ewoks executon

We will now benchmark the performance of the workflow by running it with and without `ppf` and with and without NumPy's internal multithreading.

## Run with default NumPy behavior (multi-threaded BLAS/CBLAS)

In [ ]:
import os

from ewoks import execute_graph

from ewoksutils.task_utils import task_inputs

inputs = [
    *task_inputs(
        id="matrixGenerationA", inputs={"rows": 8000, "cols": 8000, "fill": 0.2}
    ),
    *task_inputs(
        id="matrixGenerationB", inputs={"rows": 8000, "cols": 80, "fill": 0.1}
    ),
]

In [ ]:
%time

execute_graph(workflow, inputs=inputs, merge_outputs=True)

Now let's execute the workflow using the ppf engine, which enables concurrent workflow execution:

In [ ]:
%time

execute_graph(workflow, engine="ppf", inputs=inputs, merge_outputs=True)

The time difference between the two may not be significant. The reason is that, on few systems (especially laptops), NumPy's multithreading can obscure or even outperform multi-threaded parallelism due to efficient cache usage and the highly optimized nature of BLAS operations.

To truly benchmark the difference between different workflow execution engine, we will run the execution by disabling NumPy's own internal parallel processing

## Run with NumPy restricted to a single thread

We can restrict NumPy to a single thread by setting some environment variables:

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["BLIS_NUM_THREADS"] = "1"

Let's now re-execute the workflow without and with `ppf`:

In [ ]:
%time

execute_graph(workflow, inputs=inputs, merge_outputs=True)

In [ ]:
%time

execute_graph(workflow, engine="ppf", inputs=inputs, merge_outputs=True)

## Performance Notes

This workflow is structured to allow for parallel execution of the different branches. Specifically, the matrix generation and matrix transformation (transpose and flip) can run concurrently before converging at the final matrix multiplication.

However, the actual performance gain from using `ppf` depends heavily on your system:
- On laptops, the default single-core NumPy version may perform better than ppf, since the shared memory cache and uniform compute-intensive operations benefit from 
long uninterrupted CPU execution.
- On servers or multi-core machines, where processor affinity and independent caches are more favorable, the ppf engine typically performs better.

In summary, ppf provides true workflow level concurrent/parallelism, which is advantageous for heterogeneous workflows or IO-bound tasks